In [10]:
from opty import Problem, create_objective_function, parse_free
import sympy as sp
import numpy as np
import scipy as sc
import time as tm
import sympy.physics.mechanics as me
import sys 
sys.path.insert(0, "..")
from importlib import reload
import matplotlib.pyplot as plt
import equations as eq
reload (eq);
import trajectory_lib as tr
reload (tr);

participant = 'par2'
GH_seq = 'YZY'

# load OS struct created by readosim.m funcion
OS_struct = sc.io.loadmat('../Motions/'+participant+'/OS_model_prediction.mat')

# build equations of motion
# fr + frstar = 0 is implicitly defined equations of motion formed by Kane's method, 
# q, u, faux are the generalized coordinates, speeds and GH reaction forces, respectively.
q,u,faux,fr,frstar,kinematical = eq.create_eoms_quat_w_RF(OS_struct,hand_weight = 0,derive = 'numeric',gen_matlab_functions = 0)
reload(eq)
clav_pos = 0.4
tilt_y = 13
tilt_z = -6.5
w_traj = 200
w_optim_params = 1
simulation = 'All_motions'
w_optim_fmax = 5
w_optim_lceopt = 5
isim = 0
EMG_weight = 2
wGH = 2
w_act = 1
w_diff_vel = 1e-2
w_diff_exc = 1e-3
w_diff_faux = 1e-2
include_activation_dynamics = True
thor_hum_only = False

# TE is the polynomial-based spatial torque from muscle forces,
# activations is the list of symbolic muscle activations,
# TE_conoid is the spatial torque from conoid ligament
# The scalers are included as symbolic variables.
TE,activations,TE_conoid, fmax_init, fmax_range, lceopt_init, lceopt_range, mus_groups, GH_mus_forces = eq.polynomials_quat(OS_struct,q,u,calibrated_params = 'calibrate_params', derive = 'numeric', RC_lim = 1.0)
params_init = {**fmax_init, **lceopt_init}
params_range = {**fmax_range, **lceopt_range}
myKeys = list(params_init.keys())
myKeys.sort()
sd = {i: params_init[i] for i in myKeys}
sorted_params_init = np.array([*sd.values()])
num_params = len(sorted_params_init)

# one group muscle scaler scales the whole muscle group
# print(mus_groups)
# EMG names in the EMG struct and the activations that corresponds to those EMG signals in the same order.
# i.e. act_47 and act_48 are delt_clav_1 and delt_clav_2, which corresponds to the EMG signal AnteriorDelt in the EMG struct.
emg_names = ('AnteriorDelt','IntermediateDelt','PosteriorDelt','Infrasp','Suprasp','MiddleTrap','UpperTrap','Serrupper')
muscles_emg_tracked = [['act_47','act_48'],['act_43','act_44','act_45','act_46'],['act_38','act_39','act_40'],['act_56','act_57','act_58'],['act_1','act_2'],['act_6','act_7','act_8'],['act_12'],['act_26','act_27','act_28']]
# add muscle forces and conoid ligament forces to equations of motion, add kinematical differential equations and add GH muscle forces to the reaction forces
eoms_implicit = sp.Matrix(kinematical).col_join(fr+frstar+sp.Matrix([TE+sp.Matrix(TE_conoid)]).col_join(GH_mus_forces))

# if activation dynamics is included, add activation dynamics equations to the system of equations, and add excitations as specified variables
if include_activation_dynamics:
    excitations = []
    act_ode = []
    for i in range(len(activations)):
        excitations.append(me.dynamicsymbols('exc'+str(activations[i])[3:-3]))
        current_mus_ind = int(str(activations[i])[4:-3])
        current_mus = OS_struct['model']['muscles'].item()[0,(current_mus_ind-1)]
        t_act = current_mus['tact'][0,0].item()
        t_deact = current_mus['tdeact'][0,0].item()
        act_ode.append(activations[i].diff() - eq.act_dynamics(activations[i],excitations[i],t_act,t_deact))
    sp_act_ode = sp.Matrix(act_ode)
    eoms_implicit = eoms_implicit.col_join(sp_act_ode)

tasks = ['Elevation_Flexion','Scabduction_Flexion','drinking_shelf_reaching']
for itask in range(len(tasks)):
    simulation = tasks[itask]
    interval_value = 0.04
    file = '../Motions/' + participant + '/' + simulation + '/' + simulation
    traj_original,omega, num_nodes, time = tr.exp_trajectory_quat(file,interval_value)
    q0_t0 = traj_original[:,0][:4]
    traj = tr.exp_trajectory_quat_myobj(traj_original,clav_pos)
    emg, indexes_emg = tr.exp_emg('../Motions/'+participant+'/'+simulation+'/EMG_'+participant+'_'+simulation+'.mat', num_nodes = num_nodes, emg_names = emg_names)

    # track clavicle and scapula in all nodes
    index_clav_scap = 1
    # track humerus only in the nodes where EMG data is defined, len(indexes_emg) = num_nodes
    index_hum = indexes_emg

    if include_activation_dynamics:
        state_symbols = tuple(q+u+faux+activations)
        specified_symbols = tuple(excitations)
    else:
        state_symbols = tuple(q+u+faux)
        specified_symbols = tuple(activations)

    num_states = len(state_symbols) 
    num_q = len(q)
    num_u = len(u)
    num_faux = len(faux)
    num_inputs = len(specified_symbols)
    t = me.dynamicsymbols._t
    reload(eq)
    # create objective functions and their jacobians
    # trajectory tracking and clavicular orientation at t0
    objective_traj,objective_traj_jac, objective_SC_t0, objective_SC_t0_jac = eq.objective_traj_quat(num_q,interval_value,clav_pos,True)

    objective_act,objective_act_jac = eq.objective_min_activation(activations,interval_value,True,muscles_emg_tracked)

    objective_exc,objective_exc_jac = eq.objective_min_activation(activations,interval_value,True,muscles_emg_tracked)

    objective_maxstab, objective_maxstab_jac = eq.objective_max_GH_stab(tilt_y=tilt_y,tilt_z=tilt_z,interval_value = interval_value)

    obj_min_diff,obj_min_diff_jac = eq.objective_state_diff(num_nodes,interval_value)

    obj_cal_params, obj_cal_params_jac = eq.objective_scalers_restriction(mus_groups,w_optim_fmax,w_optim_lceopt)
    # indexes are defined because of the way the objective function is defined
    # we minimize activation squared, we do not minimize excitation
    w_min_squared_act = 1
    w_min_squared_exc = 0

    if include_activation_dynamics:
        w_min_emg_act = 0
        w_min_emg_exc = EMG_weight
        # if activation dynamics is included, match EMG with corresponding excitations
    else:
        # if we do not have activation dynamics, we match activations directly
        w_min_emg_act = EMG_weight
        w_min_emg_exc = 0


    def obj(free):
        min_traj = w_traj * np.sum(objective_traj(np.split(free[:num_q*num_nodes],num_q),traj,index_clav_scap,index_hum))
        min_SC_t0 = w_traj * np.sum(objective_SC_t0(free[0::num_nodes][:4],q0_t0))

        min_vel_dif = w_diff_vel * np.sum((obj_min_diff(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))))
        min_faux_dif = w_diff_faux * np.sum((obj_min_diff(np.transpose(np.split(free[(num_q+num_u)*num_nodes:(num_q + num_u+num_faux)*num_nodes],num_faux)))))

        min_act = w_act * np.sum(objective_act(np.split(free[(num_q + num_u + num_faux)*num_nodes:(num_q + num_u + num_faux + num_inputs)*num_nodes],num_inputs),emg,indexes_emg,w_min_squared_act,w_min_emg_act))

        min_instab = wGH * np.sum(objective_maxstab(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_faux)*num_nodes],3)))

        cal_params = w_optim_params * obj_cal_params(*free[(num_states + num_inputs)*num_nodes:(num_states + num_inputs + num_params)*num_nodes])

        obj = (min_traj + min_vel_dif + min_act + cal_params + min_instab + min_faux_dif + min_SC_t0) #   

        if include_activation_dynamics:
            min_exc_dif = w_diff_exc * np.sum((obj_min_diff(np.transpose(np.split(free[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes],num_inputs)))))
            min_exc = w_act * np.sum(objective_exc(np.split(free[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes],num_inputs),emg,indexes_emg,w_min_squared_exc,w_min_emg_exc))
            obj += (min_exc_dif + min_exc)

        return obj.item()

    def obj_grad(free):
        grad = np.zeros_like(free)

        grad[:num_q*num_nodes] += w_traj * np.concatenate(objective_traj_jac(np.split(free[:num_q*num_nodes],num_q),traj,index_clav_scap,index_hum))
        grad[0::num_nodes][:4] += w_traj * np.sum(objective_SC_t0_jac(free[0::num_nodes][:4],q0_t0))

        grad[(num_q + num_u + num_faux)*num_nodes:(num_q + num_u + num_faux + num_inputs)*num_nodes] += w_act * np.concatenate(objective_act_jac(np.split(free[(num_q + num_u + num_faux)*num_nodes:(num_q + num_u + num_faux + num_inputs)*num_nodes],num_inputs),emg,indexes_emg,w_min_squared_act,w_min_emg_act))
        grad[num_q*num_nodes:(num_q + num_u)*num_nodes] += w_diff_vel * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))[0,:,:])))
        grad[(num_q+num_u)*num_nodes:(num_q + num_u+num_faux)*num_nodes] += w_diff_faux * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_q+num_u)*num_nodes:(num_q + num_u+num_faux)*num_nodes],num_faux)))[0,:,:])))

        grad[(num_q + num_u)*num_nodes:(num_q + num_u + num_faux)*num_nodes] += wGH * np.concatenate(objective_maxstab_jac(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_faux)*num_nodes],3)))

        grad[(num_states + num_inputs)*num_nodes:(num_states + num_inputs + num_params)*num_nodes] += w_optim_params * obj_cal_params_jac(*free[(num_states + num_inputs)*num_nodes:(num_states + num_inputs + num_params)*num_nodes])[0]

        if include_activation_dynamics:
            grad[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes] += w_diff_exc * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes],num_inputs)))[0,:,:])))
            grad[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes] += w_act * np.concatenate(objective_exc_jac(np.split(free[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes],num_inputs),emg,indexes_emg,w_min_squared_exc,w_min_emg_exc))

        return grad

    # check objective and gradient values at a random point
    print('obj_check', obj(np.ones(num_states*num_nodes + num_inputs*num_nodes + num_params)*0.01))
    print('obj_grad_check', sum(obj_grad(np.ones(num_states*num_nodes + num_inputs*num_nodes + num_params)*0.01)))

    instance_constraints = []
    # Velocities are set zero at the first and last node.
    for i in range(num_u):
        instance_constraints.append(u[i].func(time[-1]))
        instance_constraints.append(u[i].func(time[0]))

    # unit norm of quaternions at the first node
    instance_constraints.append(state_symbols[0].func(0)**2 + state_symbols[1].func(0)**2 + state_symbols[2].func(0)**2 + state_symbols[3].func(0)**2 - 1) # SC
    instance_constraints.append(state_symbols[4].func(0)**2 + state_symbols[5].func(0)**2 + state_symbols[6].func(0)**2 + state_symbols[7].func(0)**2 - 1) # AC
    instance_constraints.append(state_symbols[8].func(0)**2 + state_symbols[9].func(0)**2 + state_symbols[10].func(0)**2 + state_symbols[11].func(0)**2 - 1) # GH

    # bounds for activations and excitations
    bounds_act = (0.0,1.0)
    bounds = (bounds_act,)*len(activations)
    bndrs = dict(zip(activations,bounds))
    if include_activation_dynamics:
        bndrs_exc = dict(zip(excitations,(bounds_act,)*len(excitations)))
        bndrs.update(bndrs_exc)

    # bounds for states and inputs, we are tracking fully, so we can set tight bounds
    for i in range(num_q):
        bndrs.update({q[i]: (min(traj_original[i,:])-0.05, max(traj_original[i,:])+0.05)})

    bndrs.update({faux[0]: (-2,0)})
    bndrs.update(params_range)


    start = tm.time()
    prob = Problem(obj, obj_grad, eoms_implicit, state_symbols,
                num_nodes, interval_value,
                known_parameter_map={},
                instance_constraints=instance_constraints,
                bounds=bndrs,
                integration_method='midpoint',
                parallel = False)


    time_to_create = tm.time() - start
    print(time_to_create)

    prob.add_option('limited_memory_max_history', 40)
    prob.add_option('acceptable_tol', 1e-4)
    prob.add_option('warm_start_init_point', 'yes')
    prob.add_option('warm_start_bound_push', 1e-3) #1e-6
    prob.add_option('warm_start_mult_bound_push', 1e-3) #1e-6
    prob.add_option('mu_init', 1e-2) # Start with a tighter barrier parameter 1e-4
    initial_guess = np.ones(prob.num_free)*0.0
    time_2_solve_start = tm.time()

    prob.add_option('max_iter',2000)
    initial_guess[:13*num_nodes] = traj_original.flatten()
    initial_guess[(num_q + num_u)*num_nodes:(num_q + num_u + 1)*num_nodes] = -0.75
    initial_guess[num_q * num_nodes : (num_q + num_u) * num_nodes] = omega.flatten()

    initial_guess[(num_states+num_inputs)*num_nodes:(num_states+num_inputs+num_params)*num_nodes] = np.ones(num_params)
        
    solution, info = prob.solve(initial_guess)
    time_2_solve = tm.time() - time_2_solve_start
    print(info['status_msg'])
    print(info['obj_val'])
    act_obj = np.sum(solution[num_states*num_nodes:(num_states + num_inputs)*num_nodes]**2)
    objective_value = prob.obj_value
    print('Objective activations: ', act_obj)

    # save result to matlab struct
    file_name = '../Motions/'+participant+'/'+simulation+'/' + simulation + '_params_calibration.mat'
    tr.sol2struct(solution,activations,num_q,num_u,num_faux,num_inputs,num_nodes,time,objective_value,time_2_solve,file_name,include_activation_dynamics)

    # save calibrated parameters to a separate mat file. The params are included in solution as last (num_scalers) values, ordered alphabetically.
    calibrated_params = dict(zip(myKeys, solution[(num_states + num_inputs)*num_nodes:(num_states + num_inputs + num_params)*num_nodes]))
    sc.io.savemat('../Motions/'+participant+'/'+simulation+'/' + simulation + '_calibrated_params.mat', {'calibrated_params': calibrated_params})

obj_check 4553.11960410519
obj_grad_check -1768.8447882210721
894.7175896167755
This is Ipopt version 3.14.19, running with linear solver MUMPS 5.7.3.

Number of nonzeros in equality constraint Jacobian...: 15227816
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        0

Total number of variables............................:    49012
                     variables with only lower bounds:        0
                variables with lower and upper bounds:    46912
                     variables with only upper bounds:        0
Total number of equality constraints.................:    26645
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  

In [2]:
from opty import Problem, create_objective_function, parse_free
import sympy as sp
import numpy as np
import scipy as sc
import time as tm
import pickle
import sympy.physics.mechanics as me
import sys
sys.path.insert(0, "..")
from importlib import reload
import matplotlib.pyplot as plt
import equations as eq
reload (eq);
import trajectory_lib as tr
reload (tr);

participant = 'par2'
GH_seq = 'YZY'

# Validaiton simulations also uses OS_model_prediction struct as it includes the polynomials for GH reaction forces vectors
OS_struct = sc.io.loadmat('../Motions/'+participant+'/OS_model_prediction.mat')

act_w = 1
include_activation_dynamics = True
# q,u,faux,fr,frstar,kinematical = eq.create_eoms_quat_w_RF(OS_struct,hand_weight = 0,derive = 'numeric',gen_matlab_functions = 0)
clav_pos = 0.4
# Define inclination (around z-axis) and version (around y-axis) angles of the glenoid.
tilt_y = 13
tilt_z = -6.5
w_traj = 200
wGH = 2
simulations = ['lifting_5kg']
calib_files = ['Elevation_Scabduction','Elevation_Flexion','Scabduction_Flexion','drinking_shelf_reaching']
w_diff_vel = 1e-3
w_diff_exc = 1e-3
w_diff_faux = 1e-3
thor_hum_only = False
for simulation in simulations:
    for cur_file in calib_files:

        if simulation =='lifting_5kg':
            q,u,faux,fr,frstar,kinematical = eq.create_eoms_quat_w_RF(OS_struct,hand_weight = 5,derive = 'numeric',gen_matlab_functions = 0)
        else:
            q,u,faux,fr,frstar,kinematical = eq.create_eoms_quat_w_RF(OS_struct,hand_weight = 0,derive = 'numeric',gen_matlab_functions = 0)


        calibrated_params = sc.io.loadmat('../Motions/'+participant+'/'+cur_file+'/'+cur_file+'_calibrated_params.mat')

        TE,activations,TE_conoid, fmax_init, fmax_range, lceopt_init, lceopt_range, mus_groups, GH_mus_forces = eq.polynomials_quat(OS_struct,q,u,calibrated_params = calibrated_params, derive = 'numeric', RC_lim = 1.0)

        eoms_implicit = sp.Matrix(kinematical).col_join(fr+frstar+sp.Matrix([TE+sp.Matrix(TE_conoid)]).col_join(GH_mus_forces))

        if include_activation_dynamics:
            excitations = []
            act_ode = []
            for i in range(len(activations)):
                excitations.append(me.dynamicsymbols('exc'+str(activations[i])[3:-3]))
                current_mus_ind = int(str(activations[i])[4:-3])
                current_mus = OS_struct['model']['muscles'].item()[0,(current_mus_ind-1)]
                t_act = current_mus['tact'][0,0].item()
                t_deact = current_mus['tdeact'][0,0].item()
                act_ode.append(activations[i].diff() - eq.act_dynamics(activations[i],excitations[i]))
            sp_act_ode = sp.Matrix(act_ode)
            eoms_implicit = eoms_implicit.col_join(sp_act_ode)

        interval_value = 0.04
        file = '../Motions/' + participant + '/' + simulation + '/' + simulation
        traj_original, omega, num_nodes, time = tr.exp_trajectory_quat(file,interval_value)
        q0_t0 = traj_original[:,0][:4]
        indexes = np.ones(num_nodes)

        traj = tr.exp_trajectory_quat_myobj(traj_original,clav_pos)

        if include_activation_dynamics:
            state_symbols = tuple(q+u+faux+activations)
            specified_symbols = tuple(excitations)
        else:
            state_symbols = tuple(q+u+faux)
            specified_symbols = tuple(activations)

        num_states = len(state_symbols) 
        num_q = len(q)
        num_u = len(u)
        num_faux = len(faux)
        num_inputs = len(specified_symbols)
        t = me.dynamicsymbols._t
        
        objective_traj,objective_traj_jac, objective_SC_t0, objective_SC_t0_jac = eq.objective_traj_quat(num_q,interval_value,clav_pos,True)

        objective_act,objective_act_jac = eq.objective_min_activation(activations,interval_value)
        objective_exc,objective_exc_jac = eq.objective_min_activation(activations,interval_value)
        objective_maxstab, objective_maxstab_jac = eq.objective_max_GH_stab(tilt_y=tilt_y,tilt_z=tilt_z,interval_value=interval_value)

        obj_min_diff,obj_min_diff_jac = eq.objective_state_diff(num_nodes,interval_value)

        indexes_clav_scap = 1
        indexes_hum = 1
    

        def obj(free):
            min_traj = w_traj * np.sum(objective_traj(np.split(free[:num_q*num_nodes],num_q),traj,indexes_clav_scap,indexes_hum))
            min_SC_t0 = w_traj * np.sum(objective_SC_t0(free[0::num_nodes][:4],q0_t0))

            min_vel_dif = w_diff_vel * np.sum((obj_min_diff(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))))
            min_faux_dif = w_diff_faux * np.sum((obj_min_diff(np.transpose(np.split(free[(num_q+num_u)*num_nodes:(num_q + num_u+num_faux)*num_nodes],num_faux)))))


            min_act = act_w * np.sum(objective_act(np.split(free[(num_q + num_u + num_faux)*num_nodes:(num_q + num_u + num_faux + num_inputs)*num_nodes],num_inputs)))


            min_instab = wGH * np.sum(objective_maxstab(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_faux)*num_nodes],3)))

            obj = (min_traj + min_vel_dif + min_act + min_instab + min_faux_dif + min_SC_t0) #   

            if include_activation_dynamics:
                min_exc_dif = w_diff_exc * np.sum((obj_min_diff(np.transpose(np.split(free[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes],num_inputs)))))
                obj += (min_exc_dif)

            return obj.item()

        def obj_grad(free):
            grad = np.zeros_like(free)

            grad[:num_q*num_nodes] += w_traj * np.concatenate(objective_traj_jac(np.split(free[:num_q*num_nodes],num_q),traj,indexes_clav_scap,indexes_hum))
            grad[0::num_nodes][:4] += w_traj * np.sum(objective_SC_t0_jac(free[0::num_nodes][:4],q0_t0))

            grad[(num_q + num_u + num_faux)*num_nodes:(num_q + num_u + num_faux + num_inputs)*num_nodes] += act_w * np.concatenate(objective_act_jac(np.split(free[(num_q + num_u + num_faux)*num_nodes:(num_q + num_u + num_faux + num_inputs)*num_nodes],num_inputs)))

            grad[num_q*num_nodes:(num_q + num_u)*num_nodes] += w_diff_vel * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))[0,:,:])))
            grad[(num_q+num_u)*num_nodes:(num_q + num_u+num_faux)*num_nodes] += w_diff_faux * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_q+num_u)*num_nodes:(num_q + num_u+num_faux)*num_nodes],num_faux)))[0,:,:])))

            grad[(num_q + num_u)*num_nodes:(num_q + num_u + num_faux)*num_nodes] += wGH * np.concatenate(objective_maxstab_jac(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_faux)*num_nodes],3)))

            if include_activation_dynamics:
                grad[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes] += w_diff_exc * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes],num_inputs)))[0,:,:])))

            return grad
        
        print('obj_check', obj(np.ones(num_states*num_nodes + num_inputs*num_nodes)*0.01))
        print('obj_grad_check', sum(obj_grad(np.ones(num_states*num_nodes + num_inputs*num_nodes)*0.01)))


        instance_constraints = []
        instance_constraints.append(state_symbols[0].func(0)**2 + state_symbols[1].func(0)**2 + state_symbols[2].func(0)**2 + state_symbols[3].func(0)**2 - 1) # SC
        instance_constraints.append(state_symbols[4].func(0)**2 + state_symbols[5].func(0)**2 + state_symbols[6].func(0)**2 + state_symbols[7].func(0)**2 - 1) # AC
        instance_constraints.append(state_symbols[8].func(0)**2 + state_symbols[9].func(0)**2 + state_symbols[10].func(0)**2 + state_symbols[11].func(0)**2 - 1) # GH

            
        bounds1 = (0.0,1.0)
        bounds = (bounds1,)*len(activations)
        bndrs = dict(zip(activations,bounds))
        if include_activation_dynamics:
            bndrs_exc = dict(zip(excitations,(bounds1,)*len(excitations)))
            bndrs.update(bndrs_exc)

                    

        

        # if cur_mot =='drinking':
        #     for i in range(num_q):
        #         if i == 8:
        #             bndrs.update({q[i]: (min(traj_original[i,:])-0.1, 1.0)})
        #         else:
        #             bndrs.update({q[i]: (min(traj_original[i,:])-0.1, max(traj_original[i,:])+0.1)})

        #     for i in range(num_u):
        #         bndrs.update({u[i]: (-3.0, 3.0)})

        # else:
        for i in range(num_q):
            if i == 8:
                bndrs.update({q[i]: (min(traj_original[i,:])-0.15, 1.0)})
            else:
                bndrs.update({q[i]: (min(traj_original[i,:])-0.15, max(traj_original[i,:])+0.15)})


        bndrs.update({faux[0]: (-2,0)})


        start = tm.time()
        prob = Problem(obj, obj_grad, eoms_implicit, state_symbols,
                    num_nodes, interval_value,
                    known_parameter_map={},
                    instance_constraints=instance_constraints,
                    bounds=bndrs,
                    integration_method='midpoint',
                    parallel = False)


        time_to_create = tm.time() - start
        print(time_to_create)

        prob.add_option('limited_memory_max_history', 40)
        prob.add_option('warm_start_bound_push', 1e-4) #1e-6
        prob.add_option('warm_start_mult_bound_push', 1e-4) #1e-6
        prob.add_option('mu_init', 5e-2) # Start with a tighter barrier parameter 1e-4
        prob.add_option('acceptable_tol', 1e-4)
        initial_guess = np.ones(prob.num_free)*0.0
        time_2_solve_start = tm.time()
        prob.add_option('max_iter',2000)
        # initial_guess[:13*num_nodes] = traj_original.flatten()
        # initial_guess[num_q * num_nodes : (num_q + num_u) * num_nodes] = omega.flatten()
        # initial_guess[(num_q + num_u)*num_nodes:(num_q + num_u + 1)*num_nodes] = -0.75
        initial_guess = tr.initial_guess_from_solution('../Motions/'+participant+'/'+simulation+'/res_'+simulation+'_0.mat',prob.num_free)[:prob.num_free]



        solution, info = prob.solve(initial_guess)
        time_2_solve = tm.time() - time_2_solve_start
        print(info['status_msg'])
        print(info['obj_val'])
        act_obj = np.sum(solution[num_states*num_nodes:(num_states + num_inputs)*num_nodes]**2)
        objective_value = prob.obj_value
        print('Objective activations: ', act_obj)

        file_name = '../Motions/'+participant+'/'+simulation+'/res_' + simulation + '_' + cur_file+'.mat'

        tr.sol2struct(solution,activations,num_q,num_u,num_faux,num_inputs,num_nodes,time,objective_value,time_2_solve,file_name,include_activation_dynamics)

        file_name_mot = '../Motions/'+participant+'/'+simulation+'/res_' + simulation + '_' + cur_file+'.mot'
        tr.sol2mot_quat(solution, num_nodes, len(q), time, file_name_mot, GH_seq)

obj_check 2742.0225342883386
obj_grad_check -1557.98830386926
882.2688691616058

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.19, running with linear solver MUMPS 5.7.3.

Number of nonzeros in equality constraint Jacobian...:  6511692
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        0

Total number of variables............................:    21560
                     variables with only lower bounds:        0
                variables with lower and upper bounds:    20636
                     variables with only upper bounds:        0
Total number

In [10]:
from opty import Problem, create_objective_function, parse_free
import sympy as sp
import numpy as np
import scipy as sc
import time as tm
import pickle
import sympy.physics.mechanics as me
import sys
sys.path.insert(0, "..")
from importlib import reload
import matplotlib.pyplot as plt
import equations as eq
reload (eq);
import trajectory_lib as tr
reload (tr);

participant = 'par2'
motion_list  =  ['All_motions']
GH_seq = 'YZY'

OS_struct = sc.io.loadmat('../Motions/'+participant+'/OS_model_prediction.mat')
act_w = 1

# build equations of motion, details are in muscle parameter calibration notebook
q,u,faux,fr,frstar,kinematical = eq.create_eoms_quat_w_RF(OS_struct,hand_weight = 0,derive = 'numeric',gen_matlab_functions = 0)
print('Equations of motion created')


# we will iterate over 7 different simulations
# The first simulation is healthy case with GH stability term = 2 (the result of the first simulation is equivalent to the one in the example notebook)
# Simulation 2-6 are with GH stability term = 2,4,6,8,10 and with the supra and infra damaged (activation set to 0)
# Simulation 7 is with non-caibrated muscle parameters
# Simulations are save in order in res_SHR_0 to res_SHR_6
# wGHs = [2,2,4,6,8,10,2]
# RC_lims = [1,0.01,0.01,0.01,0.01,0.01,1]
wGHs = [10]
RC_lims = [0.000001]
RC_names = ['RClim0']
# names = ['RClim050_calib_wgh','RClim025_calib_wgh']
calib_files = ['Elevation_Scabduction','Elevation_Flexion','Scabduction_Flexion','drinking_shelf_reaching']

# Define inclination (around z-axis) and version (around y-axis) angles of the glenoid.
tilt_y = 13
tilt_z = -6.5
w_traj = 200
simulation = 'All_elevations'
w_diff_vel = 1e-2
w_diff_exc = 1e-3
w_diff_faux = 1e-2
thor_hum_only = True
clav_pos = 0.4

# List of muscles' activations corresponding to supra and infraspinatus muscle to be set to 0 during the RC-limited condition   
act_dmg = ['act_54(t)', 'act_55(t)', 'act_56(t)', 'act_57(t)', 'act_58(t)', 'act_59(t)', 'act_67(t)', 'act_68(t)', 'act_69(t)', 'act_70(t)']

for icalib in range(len(calib_files)):
    for isim in range(len(wGHs)):
        wGH = wGHs[isim]
        RC_lim = RC_lims[isim]

        calibrated_params = sc.io.loadmat('../Motions/'+participant+'/'+calib_files[icalib]+'/'+calib_files[icalib]+'_calibrated_params.mat')

        # For the last simulation, create the torque from muscles without calibrated parameters.
        # if isim == 6:
        # if isim == 0 or isim == 2:
        #     TE,activations,TE_conoid, fmax_init, fmax_range, lceopt_init, lceopt_range, mus_groups, GH_mus_forces = eq.polynomials_quat(OS_struct,q,u,calibrated_params = None, derive = 'numeric',RC_lim = RC_lim)
        # else:    
        TE,activations,TE_conoid, fmax_init, fmax_range, lceopt_init, lceopt_range, mus_groups, GH_mus_forces = eq.polynomials_quat(OS_struct,q,u,calibrated_params = calibrated_params, derive = 'numeric',RC_lim = RC_lim)


        include_activation_dynamics = True

        eoms_implicit = sp.Matrix(kinematical).col_join(fr+frstar+sp.Matrix([TE+sp.Matrix(TE_conoid)]).col_join(GH_mus_forces))

        if include_activation_dynamics:
            excitations = []
            act_ode = []
            for i in range(len(activations)):
                excitations.append(me.dynamicsymbols('exc'+str(activations[i])[3:-3]))
                current_mus_ind = int(str(activations[i])[4:-3])
                current_mus = OS_struct['model']['muscles'].item()[0,(current_mus_ind-1)]
                t_act = current_mus['tact'][0,0].item()
                t_deact = current_mus['tdeact'][0,0].item()
                act_ode.append(activations[i].diff() - eq.act_dynamics(activations[i],excitations[i],t_act,t_deact))
            sp_act_ode = sp.Matrix(act_ode)
            eoms_implicit = eoms_implicit.col_join(sp_act_ode)

        interval_value = 0.04
        file = '../Motions/' + participant + '/' + simulation + '/' + simulation
        traj_original, omega, num_nodes, time = tr.exp_trajectory_quat(file,interval_value)
        q0_t0 = traj_original[:,0][:4]
        traj = tr.exp_trajectory_quat_myobj(traj_original,clav_pos)
        emg, indexes_emg = tr.exp_emg('../Motions/'+participant+'/'+simulation+'/EMG_'+participant+'_'+simulation+'.mat', num_nodes = num_nodes)

        # We exclude clavicle and scapula from the trajectory tracking objective
        index_clav_scap = 0
        # Humerus tracking is off during the pauses to let humerus find stable neutral position. 
        index_hum = indexes_emg
        

        if include_activation_dynamics:
            state_symbols = tuple(q+u+faux+activations)
            specified_symbols = tuple(excitations)
        else:
            state_symbols = tuple(q+u+faux)
            specified_symbols = tuple(activations)

        num_states = len(state_symbols) 
        num_q = len(q)
        num_u = len(u)
        num_faux = len(faux)
        num_inputs = len(specified_symbols)
        t = me.dynamicsymbols._t
        
        objective_traj,objective_traj_jac, objective_SC_t0, objective_SC_t0_jac = eq.objective_traj_quat(num_q,interval_value,clav_pos,True)
        objective_act,objective_act_jac = eq.objective_min_activation(activations,interval_value)
        objective_exc,objective_exc_jac = eq.objective_min_activation(activations,interval_value)
        objective_maxstab, objective_maxstab_jac = eq.objective_max_GH_stab(tilt_y=tilt_y,tilt_z=tilt_z,interval_value = interval_value)
        obj_min_diff,obj_min_diff_jac = eq.objective_state_diff(num_nodes,interval_value)

        def obj(free):
            min_traj = w_traj * np.sum(objective_traj(np.split(free[:num_q*num_nodes],num_q),traj,index_clav_scap,index_hum))
            min_SC_t0 = w_traj * np.sum(objective_SC_t0(free[0::num_nodes][:4],q0_t0))
            min_SC_tf = w_traj * np.sum(objective_SC_t0(free[num_nodes-1::num_nodes][:4],q0_t0))

            min_vel_dif = w_diff_vel * np.sum((obj_min_diff(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))))
            min_faux_dif = w_diff_faux * np.sum((obj_min_diff(np.transpose(np.split(free[(num_q+num_u)*num_nodes:(num_q + num_u+num_faux)*num_nodes],num_faux)))))

            min_act = act_w * np.sum(objective_act(np.split(free[(num_q + num_u + num_faux)*num_nodes:(num_q + num_u + num_faux + num_inputs)*num_nodes],num_inputs)))

            min_instab = wGH * np.sum(objective_maxstab(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_faux)*num_nodes],3)))

            obj = (min_traj + min_vel_dif + min_act + min_instab + min_faux_dif + min_SC_t0 + min_SC_tf) #   
            if include_activation_dynamics:
                min_exc_dif = w_diff_exc * np.sum((obj_min_diff(np.transpose(np.split(free[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes],num_inputs)))))
                obj += (min_exc_dif)

            return obj.item()

        def obj_grad(free):
            grad = np.zeros_like(free)
            grad[:num_q*num_nodes] += w_traj * np.concatenate(objective_traj_jac(np.split(free[:num_q*num_nodes],num_q),traj,index_clav_scap,index_hum))
            grad[0::num_nodes][:4] += w_traj * np.sum(objective_SC_t0_jac(free[0::num_nodes][:4],q0_t0))
            grad[num_nodes-1::num_nodes][:4] += w_traj * np.sum(objective_SC_t0_jac(free[num_nodes-1::num_nodes][:4],q0_t0))

            grad[(num_q + num_u + num_faux)*num_nodes:(num_q + num_u + num_faux + num_inputs)*num_nodes] += act_w * np.concatenate(objective_act_jac(np.split(free[(num_q + num_u + num_faux)*num_nodes:(num_q + num_u + num_faux + num_inputs)*num_nodes],num_inputs)))

            grad[num_q*num_nodes:(num_q + num_u)*num_nodes] += w_diff_vel * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[num_q*num_nodes:(num_q + num_u)*num_nodes],num_u)))[0,:,:])))
            grad[(num_q+num_u)*num_nodes:(num_q + num_u+num_faux)*num_nodes] += w_diff_faux * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_q+num_u)*num_nodes:(num_q + num_u+num_faux)*num_nodes],num_faux)))[0,:,:])))
            grad[(num_q + num_u)*num_nodes:(num_q + num_u + num_faux)*num_nodes] += wGH * np.concatenate(objective_maxstab_jac(np.split(free[(num_q + num_u)*num_nodes:(num_q + num_u + num_faux)*num_nodes],3)))

            if include_activation_dynamics:
                grad[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes] += w_diff_exc * np.concatenate(np.transpose((obj_min_diff_jac(np.transpose(np.split(free[(num_states)*num_nodes:(num_states + num_inputs)*num_nodes],num_inputs)))[0,:,:])))

            return grad
        
        print('obj_check', obj(np.ones(num_states*num_nodes + num_inputs*num_nodes)*0.01))
        print('obj_grad_check', sum(obj_grad(np.ones(num_states*num_nodes + num_inputs*num_nodes)*0.01)))

        instance_constraints = []
            
        # Velocities are set zero at the first and last node.
        for i in range(num_u):
            instance_constraints.append(u[i].func(time[-1]))
            instance_constraints.append(u[i].func(time[0])) 

        instance_constraints.append(state_symbols[0].func(0)**2 + state_symbols[1].func(0)**2 + state_symbols[2].func(0)**2 + state_symbols[3].func(0)**2 - 1) # SC
        instance_constraints.append(state_symbols[4].func(0)**2 + state_symbols[5].func(0)**2 + state_symbols[6].func(0)**2 + state_symbols[7].func(0)**2 - 1) # AC
        instance_constraints.append(state_symbols[8].func(0)**2 + state_symbols[9].func(0)**2 + state_symbols[10].func(0)**2 + state_symbols[11].func(0)**2 - 1) # GH
            
        bounds1 = (0.0,1.0)
        bounds = (bounds1,)*len(activations)
        bndrs = dict(zip(activations,bounds))
        if include_activation_dynamics:
            bndrs_exc = dict(zip(excitations,(bounds1,)*len(excitations)))
            bndrs.update(bndrs_exc)


        # for iact in activations:
        #     if str(iact) in act_dmg:
        #         if isim == 0 or isim == 6:
        #             bndrs.update({iact: (0.0, 1.0)})
        #         else:
        #             # Set the bounds of the damaged muscles' activations to 0 for simulations 2-6
        #             bndrs.update({iact: (0.0, 0.0)})
                    
        for i in range(num_q):
            if i == 0 or i == 4 or i == 8:
                bndrs.update({q[i]: (min(traj_original[i,:])-0.2, 1.0)})
            else:
                bndrs.update({q[i]: (min(traj_original[i,:])-0.2, max(traj_original[i,:])+0.2)})

        bndrs.update({faux[0]: (-2,0)})

        start = tm.time()
        prob = Problem(obj, obj_grad, eoms_implicit, state_symbols,
                    num_nodes, interval_value,
                    known_parameter_map={},
                    instance_constraints=instance_constraints,
                    bounds=bndrs,
                    integration_method='midpoint',
                    parallel = False)


        time_to_create = tm.time() - start
        print(time_to_create)
        
        # Apart from these two options, we use the default settings of the solver.
        prob.add_option('limited_memory_max_history', 40)
        prob.add_option('max_iter',2000)
        prob.add_option('warm_start_init_point', 'yes')
        prob.add_option('warm_start_bound_push', 1e-4) #1e-6
        prob.add_option('warm_start_mult_bound_push', 1e-4) #1e-6
        prob.add_option('mu_init', 5e-2) # Start with a tighter barrier parameter 1e-4
        prob.add_option('acceptable_tol', 1e-4)
        
        # Initial guesses were set such the problm converges to feasible solution as it is very senstivite to the initial guess.
        # if isim == 1:
        #     # For the RC-limited case with w_GH = 2, we use the trajectory and speeds from the experimental data, and set the faux force to -0.75 and the activations to the solution of the healthy case with w_GH = 2 as initial guess.
        #     initial_guess = np.ones(prob.num_free)*0.0
        #     initial_guess[:13*num_nodes] = traj_original.flatten()
        #     initial_guess[(num_q + num_u)*num_nodes:(num_q + num_u + 1)*num_nodes] = -0.75
        #     initial_guess[num_q * num_nodes : (num_q + num_u) * num_nodes] = omega.flatten()
        # elif isim == 0:
        #     # For the healthy case, we use the solution of the calibration problem as initial guess.
        #     initial_guess = tr.initial_guess_from_solution('../Motions/'+participant+'/'+simulation+'/All_motions_params_calibration.mat',prob.num_free)[:prob.num_free]
        # else:
        #     # For each next RC-limited case in the sensitivity analysis we use the previous solution as intial guess.
        #     initial_guess = tr.initial_guess_from_solution('../Motions/'+participant+'/'+simulation+'/res_SHR_' + str(isim-1) + '.mat',prob.num_free)[:prob.num_free]
        initial_guess = tr.initial_guess_from_solution('../Motions/'+participant+'/'+simulation+'/res_SHR_RClim050_calib_wgh2.mat',prob.num_free)[:prob.num_free]
        wGH = wGHs[isim]
        time_2_solve_start = tm.time()
        solution, info = prob.solve(initial_guess)
        time_2_solve = tm.time() - time_2_solve_start
        print(info['status_msg'])
        print(info['obj_val'])
        act_obj = np.sum(solution[num_states*num_nodes:(num_states + num_inputs)*num_nodes]**2)
        objective_value = prob.obj_value
        print('Objective activations: ', act_obj)

        # Save to matlab struct.
        # file_name = '../Motions/'+participant+'/'+simulation+'/res_SHR_' + str(isim) + '.mat'
        file_name = '../Motions/'+participant+'/'+simulation+'/res_SHR_' + RC_names[isim] +'_'+ calib_files[icalib] + '_wGH' + str(wGH) + '.mat'
        tr.sol2struct(solution,activations,num_q,num_u,num_faux,num_inputs,num_nodes,time,objective_value,time_2_solve,file_name,include_activation_dynamics)
        # Save to .mot file.
        # file_name_mot = '../Motions/'+participant+'/'+simulation+'/res_SHR_' + str(isim) + '.mot'
        # file_name_mot = '../Motions/'+participant+'/'+simulation+'/res_SHR_' + RC_names[isim] +'_'+ calib_files[icalib] + '_wGH' + str(wGH) + '.mot'
        # tr.sol2mot_quat(solution, num_nodes, len(q), time, file_name_mot, GH_seq)

Equations of motion created
obj_check 7009.294123243611
obj_grad_check -1942.638723767281
936.3837909698486
This is Ipopt version 3.14.19, running with linear solver MUMPS 5.7.3.

Number of nonzeros in equality constraint Jacobian...: 22876592
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        0

Total number of variables............................:    75040
                     variables with only lower bounds:        0
                variables with lower and upper bounds:    71824
                     variables with only upper bounds:        0
Total number of equality constraints.................:    40874
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||